# core.dom.embedding.object

> Recursive summary embeddings with raw evidence and an object relationship catalog.

Index nonempty object `summary` fields and the document summary in `*_semantics.json`.
Chroma holds summary vectors; SQLite holds exact source content, stable logical IDs,
source IDs, canonical/alternate paths, ordered parent/child relationships, and section context.
Section/prose `s` fields are context rather than additional vector records.

Copies in DOM wrappers, table headers, cells and subnodes become one logical record.
An object's source ID is scoped to its normalized semantics-file path. Moving a source
creates a different document identity. Repeated equal summaries remain separate objects.

For corpus configuration, manual indexing, live concurrency status, and evidence
retrieval, use [the object embedding application](05.application.embedding.object.ipynb).


In [ ]:
# | default_exp core.dom.embedding.object

## Canonical records

`collect_object_records` is an offline, non-mutating traversal. Paths begin at `$` and
escape `/` and `~` in keys. Owning `children` records precede `cells`, which precede
header/wrapper mirrors. Conflicting type/content values for the same ID fail visibly.
The preferred summary wins; empty summaries stay in the catalog without a vector.
The document retains the original JSON text; object `content` retains its exact source
markup. `attributes` preserves captions, line ranges, image links, and list/table details.


In [ ]:
# | export
from __future__ import annotations

import asyncio
import hashlib
import json
import math
import sqlite3
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any
from uuid import uuid4

from ribosome.core.dom.embedding.natural_text import get_embeddings_with_retry


def _json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))


def _digest(value):
    return hashlib.sha256(_json(value).encode('utf-8')).hexdigest()


def _pointer(path, key):
    return path + '/' + str(key).replace('~', '~0').replace('/', '~1')


def _heading_text(value):
    if isinstance(value, dict):
        if value.get('t') == 'Str':
            return str(value.get('c', ''))
        if value.get('t') in {'Space', 'SoftBreak', 'LineBreak'}:
            return ' '
        return _heading_text(value.get('c'))
    if isinstance(value, list):
        return ''.join(_heading_text(item) for item in value)
    return ''


@dataclass
class ObjectRecord:
    """A logical object; IDs reference other catalog records, not Chroma rows."""
    id: str
    document_id: str
    source_id: str | int | None
    type: str
    object_path: str
    occurrence_paths: list[str]
    semantics_file: str
    source_file: str
    summary: str
    content: str
    content_format: str
    parent_id: str | None
    children_ids: list[str] = field(default_factory=list)
    section_headings: list[str] = field(default_factory=list)
    attributes: dict[str, Any] = field(default_factory=dict)


@dataclass
class IndexingResult:
    semantics_file: str
    status: str
    document_id: str = ''
    generation_id: str = ''
    objects: int = 0
    embedded: int = 0
    reused: int = 0
    skipped: int = 0
    error: str = ''


@dataclass
class ObjectSearchHit:
    distance: float
    record: ObjectRecord
    ancestors: list[ObjectRecord]
    children: list[ObjectRecord]

    @property
    def summary(self):
        return self.record.summary


def collect_object_records(tree: dict, semantics_file: str | Path, *, raw_json: str | None = None) -> list[ObjectRecord]:
    """Collect one record per typed source ID without changing the input tree.

    children precede cells; wrapper subnodes and headers are mirrors. Structural
    containment determines relationships. Paths start at $ and escape / and ~.
    """
    source = str(Path(semantics_file).resolve())
    if not isinstance(tree, dict):
        raise ValueError(f'{source}: semantic JSON must be an object')
    document_id = _digest(['document', source])
    occurrences = {}

    def scan(node, path='$', parent=None, headings=(), penalty=0):
        if isinstance(node, list):
            for i, child in enumerate(node):
                scan(child, _pointer(path, i), parent, headings, penalty)
            return
        if not isinstance(node, dict):
            return
        if node.get('t') == 'Section':
            content = node.get('c', [])
            header = content[0] if isinstance(content, list) and content else {}
            title = _heading_text(header).strip()
            if title:
                headings = (*headings, title)
        if 'id' in node and isinstance(node.get('type'), str) and ('summary' in node or 'content' in node):
            identifier = node['id']
            if isinstance(identifier, bool) or not isinstance(identifier, (str, int)):
                raise ValueError(f'{source} at {path}: object id must be a string or integer')
            key = _json(identifier)
            occurrences.setdefault(key, []).append((penalty, path, parent, headings, node))
            parent = key
        # Fixed priority is independent of source dictionary insertion order.
        keys = sorted(node, key=lambda k: (0 if k == 'children' else 1 if k == 'cells' else 2, k))
        for key in keys:
            if key in {'summary', 's'}:
                continue
            extra = 1 if key == 'cells' else 2 if key == 'headers' else 0
            if key == 'subnodes' and isinstance(node.get('object'), dict):
                extra += 10
            scan(node[key], _pointer(path, key), parent, headings, penalty + extra)

    scan(tree)
    selected = {key: min(items, key=lambda item: item[0]) for key, items in occurrences.items()}
    ids = {key: _digest(['object', document_id, key]) for key in selected}

    def summary(value, path):
        if not isinstance(value, str):
            raise ValueError(f'{source} at {path}: summary must be a string')
        return value

    document = ObjectRecord(
        id=document_id, document_id=document_id, source_id=None, type='document',
        object_path='$', occurrence_paths=['$'], semantics_file=source,
        source_file=str(tree.get('file_path') or ''), summary=summary(tree.get('summary', ''), '$'),
        content=raw_json if raw_json is not None else _json(tree), content_format='json',
        parent_id=None, attributes={'title': tree.get('title', '')})
    records = [document]
    for key, (_, path, parent, headings, node) in selected.items():
        for _, other_path, _, _, other in occurrences[key]:
            if other['type'] != node['type'] or other.get('content', '') != node.get('content', ''):
                raise ValueError(f'{source}: conflicting object {node["id"]!r} at {path} and {other_path}')
        content = node.get('content', '')
        if not isinstance(content, str):
            raise ValueError(f'{source} at {path}: object content must be a string')
        records.append(ObjectRecord(
            id=ids[key], document_id=document_id, source_id=node['id'], type=node['type'],
            object_path=path, occurrence_paths=list(dict.fromkeys(item[1] for item in occurrences[key])),
            semantics_file=source, source_file=str(node.get('source_file') or tree.get('file_path') or ''),
            summary=summary(node.get('summary', ''), path), content=content, content_format='source',
            parent_id=ids[parent] if parent is not None else document_id,
            section_headings=list(headings),
            attributes={k: v for k, v in node.items() if k not in {'children', 'cells', 'summary', 'content'}}))
    by_id = {record.id: record for record in records}
    for record in records[1:]:
        if record.parent_id == record.id:
            raise ValueError(f'{source} at {record.object_path}: object contains itself')
        by_id[record.parent_id].children_ids.append(record.id)
        seen = {record.id}
        parent = record.parent_id
        while parent is not None:
            if parent in seen:
                raise ValueError(f'{source}: cyclic object relationships at {record.object_path}')
            seen.add(parent)
            parent = by_id[parent].parent_id
    # Rows may mirror only rich cells in children, while cells contains every
    # column. Relationships follow column order rather than mirror discovery.
    for record in records:
        if record.type == 'table_row':
            record.children_ids.sort(key=lambda identifier: (
                by_id[identifier].attributes.get('column', float('inf'))
                if isinstance(by_id[identifier].attributes.get('column'), int)
                else float('inf')))
    return records


## Persistent index and context search

`ObjectEmbeddingIndex` accepts injected clients/collections for testing. Its defaults
are a local Ollama client with `qwen3-embedding:8b`, a dedicated cosine `object_summaries` collection,
and `db/object_embeddings/qwen3-embedding-8b/catalog.sqlite3`. `embedding_config` identifies provider or
embedding-setting differences (for example `{"provider": "http://localhost:11434"}`);
use a new index when that configuration or the model changes. The client must be
configured to match this identity. Never mix object and natural-text vector collections.

Each successful file publishes a new immutable generation through a SQLite transaction.
Search only considers active generations and resolves context from the same generation.
Provider/storage failure or cancellation leaves the previous generation searchable.
Computed vectors are cached by summary and configuration for retries and metadata edits.
Unchanged files require no embedding calls. Inactive generations and interrupted Chroma
rows are retained for later maintenance. Source files and summary checkpoints are untouched.

`embedding_concurrency` bounds concurrent unique-summary requests within one document (default 1).
The application notebook sets it to the combined server capacity. Vectors are cached as
requests finish, then a single writer publishes the complete document. Failed or
cancelled preparation cancels and awaits all remaining workers.

Use one indexing writer per catalog; calls on one instance are serialized. `index_file`
raises errors; `index_paths` reports per-file failures and continues, but propagates
cancellation. Missing input roots fail visibly. `get_object` also returns objects with
empty summaries. `search` returns cosine distance, full records, root-first ancestors,
and immediate children, with optional document/type filters. Distances are not probabilities.
Call `close()` or use the index as a context manager to release SQLite resources.


In [ ]:
# | export
class ObjectEmbeddingIndex:
    """Summary vectors plus a versioned SQLite object catalog.

    One writer per index is supported. Queries resolve an immutable active
    generation snapshot, including context. Inactive generations are retained.
    Changing model, provider identity/configuration or collection requires a new
    index. Input semantics and summary checkpoints are never written.
    """

    def __init__(self, catalog_path: str | Path = 'db/object_embeddings/qwen3-embedding-8b/catalog.sqlite3', *,
                 collection=None, llm_client=None, model: str = 'qwen3-embedding:8b',
                 embedding_config: dict | None = None, embedding_concurrency: int = 1):
        self.catalog_path = Path(catalog_path).resolve()
        self.catalog_path.parent.mkdir(parents=True, exist_ok=True)
        if not isinstance(model, str) or not model.strip():
            raise ValueError('model must be a nonempty string')
        if isinstance(embedding_concurrency, bool) or not isinstance(embedding_concurrency, int) or embedding_concurrency < 1:
            raise ValueError('embedding_concurrency must be a positive integer')
        self.embedding_concurrency = embedding_concurrency
        self.model = model
        self.embedding_config = dict(embedding_config or {})
        self.fingerprint = _digest({'schema': 1, 'model': model, 'config': self.embedding_config})
        if llm_client is None:
            from ollama import AsyncClient
            llm_client = AsyncClient()
        self.llm_client = llm_client
        if collection is None:
            from chromadb import PersistentClient
            self.db_client = PersistentClient(path=str(self.catalog_path.parent / 'chroma'))
            collection = self.db_client.get_or_create_collection(
                'object_summaries', metadata={'hnsw:space': 'cosine'}, embedding_function=None)
        self.collection = collection
        metadata = dict(collection.metadata or {})
        configuration = getattr(collection, 'configuration', {}) or {}
        metric = metadata.get('hnsw:space') or (configuration.get('hnsw') or {}).get('space')
        if metric != 'cosine':
            raise ValueError('Object embeddings require a dedicated cosine Chroma collection')
        self.connection = sqlite3.connect(self.catalog_path)
        self.connection.row_factory = sqlite3.Row
        self._writer = asyncio.Lock()
        try:
            self.connection.executescript('''
                CREATE TABLE IF NOT EXISTS settings (key TEXT PRIMARY KEY, value TEXT NOT NULL);
                CREATE TABLE IF NOT EXISTS generations (
                    id TEXT PRIMARY KEY, document_id TEXT NOT NULL, source TEXT NOT NULL,
                    signature TEXT NOT NULL, embedded_count INTEGER NOT NULL);
                CREATE TABLE IF NOT EXISTS active_documents (
                    document_id TEXT PRIMARY KEY, generation_id TEXT NOT NULL);
                CREATE TABLE IF NOT EXISTS objects (
                    generation_id TEXT NOT NULL, id TEXT NOT NULL, record TEXT NOT NULL,
                    vector_id TEXT, PRIMARY KEY (generation_id, id));
                CREATE TABLE IF NOT EXISTS vectors (
                    cache_key TEXT PRIMARY KEY, vector TEXT NOT NULL, actual_model TEXT NOT NULL);
            ''')
            settings = dict(self.connection.execute('SELECT key, value FROM settings').fetchall())
            identity = {'fingerprint': self.fingerprint, 'collection_id': str(collection.id)}
            if settings and json.loads(settings['identity']) != identity:
                raise ValueError('Catalog embedding configuration or Chroma collection mismatch; use a new index')
            catalog_id = json.loads(settings['catalog_id']) if settings else uuid4().hex
            expected = {'object_index_config': self.fingerprint, 'object_catalog_id': catalog_id}
            if any(key in metadata and metadata[key] != value for key, value in expected.items()):
                raise ValueError('Chroma model/configuration or object catalog mismatch; use a new collection')
            if collection.count() and not all(metadata.get(key) == value for key, value in expected.items()):
                raise ValueError('Refusing to adopt a populated collection without matching object-index provenance')
            if not settings:
                with self.connection:
                    self.connection.executemany('INSERT INTO settings VALUES (?, ?)',
                        [('identity', _json(identity)), ('catalog_id', _json(catalog_id))])
            # Own only these metadata fields; retain Chroma's distance configuration.
            if any(metadata.get(key) != value for key, value in expected.items()):
                collection.modify(metadata={**{k: v for k, v in metadata.items() if not k.startswith('hnsw:')}, **expected})
            self.dimension = json.loads(settings['dimension']) if 'dimension' in settings else None
        except BaseException:
            self.connection.close()
            raise

    def close(self):
        self.connection.close()

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()

    def _validate_vector(self, vector):
        if not isinstance(vector, (list, tuple)) or not vector:
            raise ValueError('Embedding must be a nonempty vector')
        if any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) for v in vector):
            raise ValueError('Embedding must contain finite numeric values')
        if not any(vector):
            raise ValueError('Cosine embedding must have nonzero length')
        if self.dimension is not None and len(vector) != self.dimension:
            raise ValueError(f'Embedding dimension mismatch: expected {self.dimension}, got {len(vector)}')
        if self.dimension is None:
            with self.connection:
                self.connection.execute('INSERT OR IGNORE INTO settings VALUES (?, ?)', ('dimension', _json(len(vector))))
            self.dimension = json.loads(self.connection.execute(
                "SELECT value FROM settings WHERE key='dimension'").fetchone()[0])
            if len(vector) != self.dimension:
                raise ValueError('Embedding dimension mismatch')
        return [float(v) for v in vector]

    async def _embedding(self, text, label, *, cache=True):
        key = _digest([self.fingerprint, text])
        cached = self.connection.execute('SELECT vector, actual_model FROM vectors WHERE cache_key=?', (key,)).fetchone() if cache else None
        if cached is not None:
            return self._validate_vector(json.loads(cached['vector'])), cached['actual_model'], True
        vector, response = await get_embeddings_with_retry(text, label, self.llm_client, self.model)
        vector = self._validate_vector(vector)
        actual = response.get('model') if isinstance(response, dict) else getattr(response, 'model', None)
        actual = actual if isinstance(actual, str) and actual else self.model
        if cache:
            with self.connection:
                self.connection.execute('INSERT OR REPLACE INTO vectors VALUES (?, ?, ?)', (key, _json(vector), actual))
        return vector, actual, False

    async def _prepare_embeddings(self, records):
        """Prepare summaries with bounded workers; keep vector payloads in SQLite."""
        unique = {}
        for record in records:
            if record.summary.strip():
                unique.setdefault(record.summary, f'{record.semantics_file} at {record.object_path}')
        pending = iter(unique.items())
        prepared = {}

        async def worker():
            for text, label in pending:
                _, _, was_reused = await self._embedding(text, label)
                prepared[text] = was_reused

        tasks = [asyncio.create_task(worker()) for _ in range(min(self.embedding_concurrency, len(unique)))]
        try:
            await asyncio.gather(*tasks)
        except BaseException:
            for task in tasks:
                task.cancel()
            await asyncio.gather(*tasks, return_exceptions=True)
            raise
        return prepared

    def _active_generation(self, document_id):
        row = self.connection.execute('SELECT generation_id FROM active_documents WHERE document_id=?', (document_id,)).fetchone()
        return row[0] if row else None

    async def index_file(self, semantics_file: str | Path) -> IndexingResult:
        """Publish a complete document, reusing cached vectors on rerun or retry.

        Exceptions (including cancellation) propagate; the old active generation
        remains accessible. Successfully computed vectors are cached for retry.
        """
        async with self._writer:
            source = Path(semantics_file).resolve()
            raw = source.read_bytes().decode('utf-8')
            records = collect_object_records(json.loads(raw), source, raw_json=raw)
            document_id = records[0].id
            signature = _digest([self.fingerprint, [asdict(record) for record in records]])
            previous = self._active_generation(document_id)
            if previous:
                row = self.connection.execute('SELECT signature FROM generations WHERE id=?', (previous,)).fetchone()
                if row['signature'] == signature:
                    count = sum(bool(record.summary.strip()) for record in records)
                    return IndexingResult(str(source), 'unchanged', document_id, previous,
                        len(records) - 1, 0, count, len(records) - count)
            generation = uuid4().hex
            embedded = reused = skipped = 0
            prepared = await self._prepare_embeddings(records) if self.embedding_concurrency > 1 else None
            used_summaries = set()
            catalog_rows = []
            for record in records:
                vector_id = None
                if record.summary.strip():
                    if prepared is None:
                        vector, actual, was_reused = await self._embedding(record.summary, f'{source} at {record.object_path}')
                    else:
                        vector, actual, _ = await self._embedding(record.summary, f'{source} at {record.object_path}')
                        was_reused = prepared[record.summary] or record.summary in used_summaries
                    used_summaries.add(record.summary)
                    reused += int(was_reused)
                    embedded += int(not was_reused)
                    vector_id = _digest([generation, record.id])
                    self.collection.upsert(
                        ids=[vector_id], embeddings=[vector], documents=[record.summary],
                        metadatas=[{'object_id': record.id, 'document_id': document_id,
                            'generation_id': generation, 'object_type': record.type,
                            'object_path': record.object_path, 'semantics_file': str(source),
                            'source_file': record.source_file, 'embed_model': self.model,
                            'actual_model': actual, 'config_fingerprint': self.fingerprint}])
                else:
                    skipped += 1
                catalog_rows.append((generation, record.id, _json(asdict(record)), vector_id))
            # Guard against publishing a stale read if an unsupported second writer
            # has published this document while provider requests were running.
            try:
                self.connection.execute('BEGIN IMMEDIATE')
                if self._active_generation(document_id) != previous:
                    raise RuntimeError('Another writer published this document; retry with one indexing writer')
                self.connection.execute('INSERT INTO generations VALUES (?, ?, ?, ?, ?)',
                    (generation, document_id, str(source), signature, embedded + reused))
                self.connection.executemany('INSERT INTO objects VALUES (?, ?, ?, ?)', catalog_rows)
                self.connection.execute('INSERT OR REPLACE INTO active_documents VALUES (?, ?)', (document_id, generation))
                self.connection.commit()
            except BaseException:
                self.connection.rollback()
                raise
            return IndexingResult(str(source), 'indexed', document_id, generation,
                len(records) - 1, embedded, reused, skipped)

    async def index_paths(self, paths) -> list[IndexingResult]:
        """Recursively discover *_semantics.json; report individual file failures.

        Explicit files must exist and match the suffix. Missing/invalid input roots
        fail visibly instead of reporting a misleading empty successful run.
        """
        if isinstance(paths, (str, Path)):
            paths = [paths]
        files = set()
        for value in paths:
            path = Path(value).resolve()
            if path.is_dir():
                files.update(path.rglob('*_semantics.json'))
            elif path.is_file() and path.name.endswith('_semantics.json'):
                files.add(path)
            else:
                raise ValueError(f'Expected a directory or existing *_semantics.json file: {path}')
        results = []
        for path in sorted(files):
            try:
                results.append(await self.index_file(path))
            except Exception as exc:
                results.append(IndexingResult(str(path), 'failed', error=f'{type(exc).__name__}: {exc}'))
        return results

    def _get_record(self, object_id, generation):
        row = self.connection.execute('SELECT record FROM objects WHERE id=? AND generation_id=?', (object_id, generation)).fetchone()
        return ObjectRecord(**json.loads(row[0])) if row else None

    def get_object(self, object_id: str) -> ObjectRecord | None:
        """Look up a logical ID in the currently active catalog, including unembedded objects."""
        row = self.connection.execute('''SELECT o.record FROM objects o
            JOIN active_documents a ON a.generation_id=o.generation_id WHERE o.id=?''', (object_id,)).fetchone()
        return ObjectRecord(**json.loads(row[0])) if row else None

    async def search(self, question: str, *, top_k: int = 5, document_ids=None, object_types=None) -> list[ObjectSearchHit]:
        """Cosine nearest summaries with raw evidence, ancestors and direct children.

        Filters accept sequences of IDs/types. Empty filter sequences match nothing.
        Distances are returned as distances, not probability or relevance scores.
        """
        if not isinstance(question, str) or not question.strip():
            raise ValueError('question must be a nonempty string')
        if isinstance(top_k, bool) or not isinstance(top_k, int) or top_k < 1:
            raise ValueError('top_k must be a positive integer')
        for name, values in [('document_ids', document_ids), ('object_types', object_types)]:
            if values is not None and (isinstance(values, str) or not all(isinstance(v, str) for v in values)):
                raise ValueError(f'{name} must be a sequence of strings')
            if values is not None and not values:
                return []
        snapshot = self.connection.execute('''SELECT a.document_id, a.generation_id, g.embedded_count
            FROM active_documents a JOIN generations g ON g.id=a.generation_id''').fetchall()
        if document_ids is not None:
            snapshot = [row for row in snapshot if row['document_id'] in document_ids]
        snapshot = [row for row in snapshot if row['embedded_count']]
        if not snapshot:
            return []
        vector, _, _ = await self._embedding(question, 'query', cache=False)
        candidates = []
        # Bound filter size while preserving the global top-k after merging.
        for start in range(0, len(snapshot), 100):
            batch = snapshot[start:start + 100]
            where = {'generation_id': {'$in': [row['generation_id'] for row in batch]}}
            if object_types is not None:
                where = {'$and': [where, {'object_type': {'$in': list(object_types)}}]}
            count = sum(row['embedded_count'] for row in batch)
            result = self.collection.query(query_embeddings=[vector], n_results=min(top_k, count),
                where=where, include=['distances', 'metadatas'])
            for metadata, distance in zip(result['metadatas'][0], result['distances'][0]):
                candidates.append((float(distance), metadata['object_id'], metadata['generation_id']))
        hits = []
        for distance, identifier, generation in sorted(candidates)[:top_k]:
            record = self._get_record(identifier, generation)
            if record is None:
                raise RuntimeError(f'Active Chroma row has no catalog record: {identifier}')
            ancestors = []
            parent_id = record.parent_id
            while parent_id is not None:
                parent = self._get_record(parent_id, generation)
                if parent is None:
                    raise RuntimeError(f'Missing parent catalog record: {parent_id}')
                ancestors.append(parent)
                parent_id = parent.parent_id
            children = [self._get_record(child_id, generation) for child_id in record.children_ids]
            if any(child is None for child in children):
                raise RuntimeError(f'Missing child catalog record for {identifier}')
            hits.append(ObjectSearchHit(distance, record, list(reversed(ancestors)), children))
        return hits


## Offline functional test

This end-to-end check uses an injected fake embedding provider, temporary SQLite,
and ephemeral Chroma. It verifies recursive indexing and ignored objects, source
preservation, unchanged reruns without model calls, retrieval with parent context,
and safe retry after a failed update. It never contacts Ollama or indexes a corpus.
Additional edge cases are covered in `tests/test_object_embedding.py`.

In [ ]:
# | hide
from tempfile import TemporaryDirectory
import chromadb


async def test_object_embedding_roundtrip():
    class OfflineProvider:
        def __init__(self):
            self.calls = []
            self.fail = False

        async def embed(self, model, input):
            await asyncio.sleep(0)
            if self.fail:
                raise RuntimeError('Simulated embedding failure')
            self.calls.append(input)
            return {'embeddings': [[float(len(input)), 1.0, 2.0]], 'model': model}

    tree = {'summary': 'Document', 'objects': [
        {'id': 1, 'type': 'list', 'content': '- raw item\n', 'summary': 'List', 'children': [
            {'id': '1.item.1', 'type': 'list_item', 'content': '- raw item\n', 'summary': 'Needle'},
            {'id': '1.figure.1', 'type': 'figure', 'content': '![](ignored.png)',
             'summary': '', 'summary_ignored': True}]}]}
    with TemporaryDirectory() as directory:
        source = Path(directory) / 'manual_semantics.json'
        source.write_text(json.dumps(tree), encoding='utf-8')
        original = source.read_bytes()
        database = chromadb.EphemeralClient()
        collection = database.create_collection('offline-object-test-' + uuid4().hex,
            metadata={'hnsw:space': 'cosine'}, embedding_function=None)
        provider = OfflineProvider()
        try:
            with ObjectEmbeddingIndex(Path(directory) / 'catalog.sqlite3', collection=collection,
                    llm_client=provider, model='offline-model', embedding_concurrency=2) as index:
                result = await index.index_file(source)
                assert (result.objects, result.embedded, result.skipped) == (3, 3, 1)
                assert source.read_bytes() == original
                records = collect_object_records(tree, source)
                ignored = next(record for record in records if record.type == 'figure')
                assert index.get_object(ignored.id).summary == ''
                requests = len(provider.calls)
                assert (await index.index_file(source)).status == 'unchanged'
                assert len(provider.calls) == requests
                hit, = await index.search('Needle', top_k=1, object_types=['list_item'])
                assert hit.record.content == '- raw item\n'
                assert [parent.type for parent in hit.ancestors] == ['document', 'list']
                assert hit.record.parent_id == hit.ancestors[-1].id
                tree['objects'][0]['children'][0]['summary'] = 'Updated needle'
                source.write_text(json.dumps(tree), encoding='utf-8')
                changed = source.read_bytes()
                provider.fail = True
                try:
                    await index.index_file(source)
                except RuntimeError as error:
                    assert 'Simulated embedding failure' in str(error)
                else:
                    raise AssertionError('A provider failure must prevent publication')
                assert index.get_object(hit.record.id).summary == 'Needle'
                provider.fail = False
                retried = await index.index_file(source)
                assert (retried.embedded, retried.reused) == (1, 2)
                assert index.get_object(hit.record.id).summary == 'Updated needle'
                assert source.read_bytes() == changed
        finally:
            database.delete_collection(collection.name)


_ = await test_object_embedding_roundtrip()